# SMPC Private Inference — Colab Demo

**Privacy mechanism:** 2-party additive secret sharing  
**Threat model:** semi-honest, static corruption, polynomial-time adversary  
**Dataset:** Chest X-ray Pneumonia (Kaggle)  

This notebook is **self-contained and independent** from the DP and Block-Permutation settings.

## 1 — Setup

In [ ]:
# Install Kaggle and download the dataset
!pip install -q kaggle

from google.colab import files
files.upload()   # upload your kaggle.json here

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
!unzip -q chest-xray-pneumonia.zip

In [ ]:
# Clone the project repo so privacy_ml is importable
import os, sys

REPO = "/content/Privacy_Preserving_ML"
if not os.path.exists(REPO):
    !git clone https://github.com/Erayisci/Privacy_Preserving_ML.git {REPO}

sys.path.insert(0, REPO)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from pathlib import Path

from privacy_ml.models import (
    build_encoder,
    build_head,
    build_end_to_end,
    compile_for_binary_classification,
    DEFAULT_IMG_SIZE,
    DEFAULT_CHANNELS,
    EMBEDDING_DIM,
    DEFAULT_DROPOUT_RATE,
    DEFAULT_LEARNING_RATE,
)
from privacy_ml.data import load_kaggle_pool, split_pool_indices
from privacy_ml.smpc import smpc_predict, smpc_accuracy

print("TensorFlow:", tf.__version__)
print("NumPy:", np.__version__)

## 2 — Load Data

In [ ]:
BASE_DIR = Path("/content/chest_xray")

print("Loading images into memory (this takes ~1–2 min)...")
X, y = load_kaggle_pool(BASE_DIR, img_size=DEFAULT_IMG_SIZE)
print(f"Loaded {len(X)} images — shape {X.shape}, labels {y.shape}")
print(f"Class balance (PNEUMONIA fraction): {y.mean():.2%}")

In [ ]:
SEED = 42
splits = split_pool_indices(y, seed=SEED)

X_train = X[splits.victim_members]
y_train = y[splits.victim_members]

X_test  = X[splits.victim_nonmembers]
y_test  = y[splits.victim_nonmembers]

print(f"Train: {len(X_train)} images | Test: {len(X_test)} images")

## 3 — Train the Split Model (Encoder + Head)

In [ ]:
encoder  = build_encoder(DEFAULT_IMG_SIZE, DEFAULT_CHANNELS, EMBEDDING_DIM, name="encoder")
head     = build_head(EMBEDDING_DIM, DEFAULT_DROPOUT_RATE, name="head")
pipeline = build_end_to_end(encoder, head, name="pipeline")
pipeline = compile_for_binary_classification(pipeline, DEFAULT_LEARNING_RATE)

pipeline.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

history = pipeline.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=15,
    batch_size=32,
    callbacks=[EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)],
)

# Save weights so you don't have to retrain
pipeline.save_weights("/content/pipeline_weights.weights.h5")
print("Weights saved.")

> **Skip training next time:** uncomment the cell below to reload saved weights.

In [ ]:
# encoder  = build_encoder(DEFAULT_IMG_SIZE, DEFAULT_CHANNELS, EMBEDDING_DIM, name="encoder")
# head     = build_head(EMBEDDING_DIM, DEFAULT_DROPOUT_RATE, name="head")
# pipeline = build_end_to_end(encoder, head, name="pipeline")
# pipeline = compile_for_binary_classification(pipeline, DEFAULT_LEARNING_RATE)
# pipeline.load_weights("/content/pipeline_weights.weights.h5")
# print("Weights loaded.")

## 4 — Baseline Evaluation (No Privacy)

In [ ]:
baseline_loss, baseline_acc = pipeline.evaluate(X_test, y_test, verbose=0)
print(f"Baseline  — accuracy: {baseline_acc:.4f}  |  loss: {baseline_loss:.4f}")

## 5 — SMPC Inference

**What happens here (matching the slides):**

```
Hospital side:
  x (embedding)  →  x = x1 + x2  (additive secret sharing)
  → send x1 to Server A, x2 to Server B

Server A:  partial_A = x1 @ W
Server B:  partial_B = x2 @ W + b

Reconstruction:  logit = partial_A + partial_B  =  x @ W + b
Final output:    sigmoid(logit)  →  diagnosis
```

Neither server ever sees the full embedding `x`.

In [ ]:
# Step 1: Hospital extracts embeddings (encoder runs locally at the hospital)
print("Extracting embeddings via encoder...")
embeddings = encoder.predict(X_test, verbose=1)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Step 2: Run SMPC inference
# smpc_predict() handles secret sharing + 2-server collaborative linear inference
smpc_preds = smpc_predict(embeddings, head, seed=SEED)
smpc_acc   = smpc_accuracy(smpc_preds, y_test)

print(f"SMPC      — accuracy: {smpc_acc:.4f}")

## 6 — Privacy Sanity Check

Verify that individual shares carry no information about the original embedding.

In [ ]:
from privacy_ml.smpc import additive_share, reconstruct

rng = np.random.default_rng(SEED)
share1, share2 = additive_share(embeddings, rng)

# Cosine similarity between share1 and x (should be near 0 — no correlation)
def mean_cosine_similarity(a, b):
    a_norm = a / (np.linalg.norm(a, axis=1, keepdims=True) + 1e-8)
    b_norm = b / (np.linalg.norm(b, axis=1, keepdims=True) + 1e-8)
    return float(np.mean(np.sum(a_norm * b_norm, axis=1)))

cos_s1_x  = mean_cosine_similarity(share1, embeddings)
cos_rec_x = mean_cosine_similarity(reconstruct(share1, share2), embeddings)

print(f"Cosine similarity  share1 ↔ x  : {cos_s1_x:.4f}  (should be ~0)")
print(f"Cosine similarity  recon  ↔ x  : {cos_rec_x:.4f}  (should be ~1)")

## 7 — Results Summary

In [ ]:
print("=" * 40)
print(f"  Baseline accuracy : {baseline_acc:.4f}")
print(f"  SMPC     accuracy : {smpc_acc:.4f}")
print(f"  Utility loss      : {abs(baseline_acc - smpc_acc):.6f}")
print("=" * 40)
print()
print("Expected: utility loss ≈ 0 (additive secret sharing is lossless for linear ops)")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy comparison
methods = ["Baseline\n(no privacy)", "SMPC\n(secret sharing)"]
accs    = [baseline_acc, smpc_acc]
colors  = ["steelblue", "seagreen"]

axes[0].bar(methods, accs, color=colors, width=0.4)
axes[0].set_ylim(0, 1)
axes[0].set_ylabel("Test Accuracy")
axes[0].set_title("Accuracy: Baseline vs SMPC")
for i, v in enumerate(accs):
    axes[0].text(i, v + 0.01, f"{v:.4f}", ha="center", fontweight="bold")

# Share vs original distribution (first embedding, first 20 dims)
idx = 0
dims = range(20)
axes[1].plot(dims, embeddings[idx, :20], label="Original x",  marker="o")
axes[1].plot(dims, share1[idx, :20],     label="Share 1 (x1)", marker="x", linestyle="--")
axes[1].plot(dims, share2[idx, :20],     label="Share 2 (x2)", marker="+", linestyle=":")
axes[1].set_xlabel("Embedding dimension")
axes[1].set_title("Secret Shares vs Original (first 20 dims, sample 0)")
axes[1].legend()

plt.tight_layout()
plt.savefig("/content/smpc_results.png", dpi=150)
plt.show()